# DeepEval RAG Evaluation

## Overview

[DeepEval](https://deepeval.com/) is an open-source framework for evaluating LLM applications, built around four design principles:

- **Local-first.** Evaluations run in your own environment, against the code, datasets, and traces you're actively editing.
- **Pytest-native.** Metrics score outputs on a 0–1 scale with a plain-language reason; a configurable threshold turns each score into pass or fail, which you can rerun locally or wire into CI.
- **Trace-aware.** When failures matter, traces let you see which tool call, retriever, or generator step caused the regression — not just the final output.
- **Composable.** Datasets, metrics, traces, custom models, and custom scoring logic combine as building blocks instead of one rigid pipeline.

DeepEval ships with 50+ built-in metrics spanning RAG, agents, tool use, multi-turn conversations, safety, and multimodal applications, and supports custom metrics in natural language when the defaults don't fit. When you need shared dashboards, regression tracking, or production monitoring, it integrates with [Confident AI](https://www.confident-ai.com/).

Five concepts show up throughout the framework and are worth knowing upfront:

- **Test cases** structure the inputs, outputs, expected behavior, and context you want to evaluate.
- **Datasets** organize reusable test cases for repeatable runs across prompts, models, and releases.
- **Metrics** define how outputs, traces, and spans are scored.
- **Traces and spans** capture what happened during execution so you can evaluate full runs or individual components.
- **Synthetic data generation** bootstraps a dataset from documents, seed examples, or contexts when you don't have enough real examples yet.

## The Scenario

Throughout this notebook, we'll work through the following scenario:

You're building an internal AWS documentation assistant: a user asks a question, a retriever returns documentation passages, and a model generates an answer using those passages. This notebook evaluates both stages — how good the retrieved context is, and how well the model uses it to produce an answer. Retrieval is never perfect — sometimes it returns the right passage in full, sometimes it covers only part of the question, sometimes it surfaces a topically related passage that doesn't actually answer what was asked. Your job is to evaluate both stages, diagnose where failures originate, and pick a model that holds up across that spectrum.

To decide whether the system "holds up," we evaluate two stages separately:

**Retrieval quality** — did the retriever surface useful context?

- **Relevant** — the retrieved passage is on-topic for the question.
- **Sufficient** — the passage contains enough information to produce a complete answer.

**Generation quality** — given that context, did the model produce a good answer?

- **On-topic** — the answer addresses the question rather than drifting or padding.
- **Grounded** — every claim in the answer is supported by the provided context, with nothing invented.
- **Factually correct** — the answer agrees with a known-good reference rather than confidently stating something wrong.
- **Complete** — the answer covers the essential points from the reference, not just a fraction.

Each property maps to a DeepEval metric; we'll wire them up shortly.

## What You'll Learn

This notebook walks through the RAG evaluation workflow for that scenario, using Bedrock models for both the candidates being evaluated and the judge model DeepEval uses to score them. The dataset is a 20-example set of AWS Q&A pairs designed to test different failure modes. Concretely, you'll learn how to:

- Structure an evaluation as `LLMTestCase` objects and score them with metrics
- Use `AmazonBedrockModel` so any model can serve as the DeepEval judge
- Evaluate retrieval quality separately from generation quality
- Mix built-in RAG metrics with custom G-Eval metrics for failure modes the built-ins don't cover
- Cross-reference retrieval and generation scores to diagnose where failures originate
- Compare candidate models and read the judge's reasons on failures

**What's out of scope.** Other eval modes — adversarial testing, production monitoring, CI regression gates, multi-turn evaluation — are out of scope here.

## 1. Exploring the dataset

We use `data/aws_qa.json` — 20 examples covering conceptual AWS topics across compute, storage, networking, identity, and operations. Each example has a question (`question`), a grounding passage (`context`), a reference answer (`expected_answer`), and a link to the source documentation (`source_url`). Passages are paraphrased from real AWS documentation.

The dataset is designed to exercise different failure modes. Some questions have context that fully answers them — these are the easy cases any decent model should pass. Others have context that covers only part of a two-part question — the honest answer names what isn't in the context instead of guessing. A few have context that's topically adjacent but doesn't actually answer the specific question — the honest answer declines to answer. This mix is what separates a model that blindly fills in the blanks from one that stays faithful to what it was given.

> ⚠️ **Heads-up:** 20 examples is a teaching dataset. It's enough to see patterns emerge in this notebook, but any real model-selection decision needs more examples than this, ideally drawn from your actual user traffic. We'll come back to this at the end.

### Test case shape

Every evaluation in DeepEval is built around a test case. DeepEval has several test case types for different kinds of evaluation — `LLMTestCase` for single-turn prompts, `ConversationalTestCase` for multi-turn dialogues, and others for multimodal and arena-style comparisons. This notebook uses `LLMTestCase`, which has several fields; different metrics read different subsets. The four we use most in this notebook:

| Field | What it is | In our dataset |
|---|---|---|
| `input` | What the user asked | `question` |
| `actual_output` | What the model answered | Generated by calling Bedrock |
| `expected_output` | A reference answer | `expected_answer` |
| `retrieval_context` | Retrieved passages from AWS documentation | `[context]` |

Not every field is needed every time.

> 💡 **Note:** Different metrics require different subsets of these fields. A metric that checks relevance only needs `input` and `actual_output`; a metric that checks groundedness also needs `retrieval_context`; a metric that compares against a reference needs `expected_output`. A metric given a test case missing a field it needs raises an error at measurement time.

Let's load the dataset and peek at three examples that illustrate the difficulty spectrum.

In [ ]:
import json
from pathlib import Path

DATA_PATH = Path("data/aws_qa.json")
rows = json.loads(DATA_PATH.read_text())
print(f"Loaded {len(rows)} examples.\n")

# Peek at three examples that illustrate the difficulty spectrum.
# aws_01 has context that directly answers the question.
# aws_08 has context that covers only half the question (SQS but not SNS).
# aws_16 has context that's topically adjacent but doesn't answer the specific question.
samples = {r["id"]: r for r in rows}
for rid in ["aws_01", "aws_08", "aws_16"]:
    r = samples[rid]
    print(f"--- {rid} ---")
    print(f"Q: {r['question']}")
    print(f"Context ({len(r['context'].split())} words): {r['context'][:180]}...")
    print(f"Expected: {r['expected_answer']}")
    print()

## 2. Building the evaluation harness

With the dataset in hand, we build the harness that will score candidate answers against it: install DeepEval, connect to Bedrock, wire up the models we'll use, define the metrics, and finish with a quick check to make sure everything works.

### Installing dependencies

Run the cell below to install the dependencies from `requirements.txt`.

In [ ]:
%pip install -q -r requirements.txt

### Connecting to Bedrock

All model calls in this notebook go through the [Bedrock Runtime Converse API](https://docs.aws.amazon.com/bedrock/latest/userguide/conversation-inference.html). We create a single client with retries enabled, then verify credentials by printing the AWS account and region.

In [ ]:
import boto3
from botocore.config import Config

AWS_REGION = "us-east-1"
bedrock = boto3.client(
    "bedrock-runtime",
    region_name=AWS_REGION,
    config=Config(retries={"max_attempts": 10, "mode": "standard"}),
)

# Verify credentials are set correctly
identity = boto3.client("sts").get_caller_identity()
print(f"Account: {identity['Account']}")
print(f"Region:  {AWS_REGION}")

### Wiring up the models

We use three Bedrock models, accessed through cross-region inference profiles (the `us.` prefix), which route the call to whichever US region has capacity.

| Role | Model | Description (from AWS documentation) |
|---|---|---|
| Candidate #1 | `us.amazon.nova-2-lite-v1:0` | "A cost-efficient multimodal model designed for simple automation, document processing, and customer support, supporting text, image, audio, speech, and video." |
| Candidate #2 | `us.anthropic.claude-haiku-4-5-20251001-v1:0` | "Anthropic's lightweight model, optimized for speed and efficiency with strong coding and agent performance." |
| Judge | `us.anthropic.claude-sonnet-4-6` | "Anthropic's mid-tier model, featuring improvements in coding, computer use, long-context reasoning, and agent planning." |

The candidates are the models we're *evaluating*. The judge is the model DeepEval uses *to do the evaluating*. Picking a judge that's more capable than either candidate keeps the verdict credible — you don't want your grader to be weaker than the student.

The cell below defines the model IDs, instantiates the judge with `AmazonBedrockModel` (DeepEval's built-in Bedrock wrapper), and defines a small `converse()` helper for candidate generation later. Candidate calls don't go through DeepEval — we're calling the candidate models directly to produce answers, which DeepEval then scores separately.

In [ ]:
NOVA_2_LITE_ID = "us.amazon.nova-2-lite-v1:0"
HAIKU_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
JUDGE_ID = "us.anthropic.claude-sonnet-4-6"

from deepeval.models import AmazonBedrockModel

judge = AmazonBedrockModel(
    model=JUDGE_ID,
    region=AWS_REGION,
    generation_kwargs={"temperature": 0},
)


def converse(model_id: str, prompt: str, system: str | None = None,
             temperature: float = 0, max_tokens: int = 400) -> str:
    """Call any Bedrock Converse-capable model and return the text reply."""
    body = {
        "messages": [{"role": "user", "content": [{"text": prompt}]}],
        "inferenceConfig": {"maxTokens": max_tokens, "temperature": temperature},
    }
    if system:
        body["system"] = [{"text": system}]
    response = bedrock.converse(modelId=model_id, **body)
    return response["output"]["message"]["content"][0]["text"]


# Smoke test each model
for label, model_id in [("nova_2_lite", NOVA_2_LITE_ID), ("haiku", HAIKU_ID), ("judge", JUDGE_ID)]:
    reply = converse(model_id, "Say 'ok' and nothing else.")
    print(f"  {label:10s} ({model_id}): {reply!r}")

### Defining the metrics

The success criteria we defined earlier in the scenario map onto six DeepEval metrics, split across the two RAG stages:

**Retrieval metrics:**

| Criterion | Metric | What it evaluates |
|---|---|---|
| Relevant | `ContextualRelevancyMetric` | Whether the retrieved context is relevant to the question |
| Sufficient | `ContextualRecallMetric` | Whether the retrieved context contains enough information to produce the expected answer |

We use these two metrics to evaluate retrieval quality before moving to generation.

> 💡 **Note:** DeepEval also offers `ContextualPrecisionMetric`, which evaluates whether a re-ranker ranks relevant passages above irrelevant ones. It requires multiple passages per example to be meaningful. Our dataset has one passage per example, so we skip it here. If your pipeline includes a re-ranker and returns multiple chunks, add it to your metric set.

**Generation metrics:**

| Criterion | Metric | What it evaluates |
|---|---|---|
| On-topic | `AnswerRelevancyMetric` | Whether the answer addresses the question — targets prompt template quality |
| Grounded | `FaithfulnessMetric` | Whether every claim is supported by the context — targets LLM choice and tendency to hallucinate |
| Factually correct | `Correctness` (G-Eval) | Whether the answer agrees with the reference |
| Complete | `Completeness` (G-Eval) | Whether the answer covers the key points from the reference |

The retrieval metrics and the first two generation metrics are DeepEval built-ins. Correctness and Completeness we define with [`G-Eval`](https://deepeval.com/docs/metrics-llm-evals), a framework that uses an LLM with chain-of-thought reasoning to score outputs against any custom criteria you describe in plain English. You can describe what to evaluate either as a single `criteria` string (G-Eval generates the scoring steps for you) or as an explicit list of `evaluation_steps` (the LLM follows them directly). DeepEval's documentation recommends providing your own `evaluation_steps` for more controllable metric scores.

Each metric has a `threshold` that determines whether a score passes. Metrics default to `threshold=0.5`. Whether that's the right cutoff depends on your domain; we use `0.7` below.

In [ ]:
from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    ContextualRelevancyMetric,
    ContextualRecallMetric,
    GEval,
)
from deepeval.test_case import LLMTestCase, SingleTurnParams

# --- Retrieval metrics ---
contextual_relevancy = ContextualRelevancyMetric(model=judge, threshold=0.7)
contextual_recall = ContextualRecallMetric(model=judge, threshold=0.7)

RETRIEVAL_METRICS = [contextual_relevancy, contextual_recall]
RETRIEVAL_METRIC_NAMES = ["Contextual Relevancy", "Contextual Recall"]

# --- Generation metrics (built-in) ---
answer_relevancy = AnswerRelevancyMetric(model=judge, threshold=0.7)
faithfulness = FaithfulnessMetric(model=judge, threshold=0.7)

# --- Generation metrics (custom G-Eval) ---
correctness = GEval(
    name="Correctness",
    evaluation_steps=[
        "Check whether the facts in 'actual output' contradict any facts in 'expected output'.",
        "Penalize heavily if the actual output states something that is factually wrong according to the expected output.",
        "Vague or approximate language is acceptable as long as it is not contradicted by the expected output.",
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT],
    model=judge,
    threshold=0.7,
)

completeness = GEval(
    name="Completeness",
    evaluation_steps=[
        "Identify the key points present in the 'expected output'.",
        "Check whether each key point is addressed in the 'actual output'.",
        "Penalize omissions where the actual output misses a point the expected output treats as essential.",
        "Do NOT penalize the actual output for adding relevant detail beyond the expected output - only for leaving required points out.",
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT],
    model=judge,
    threshold=0.7,
)

GENERATION_METRICS = [answer_relevancy, faithfulness, correctness, completeness]
GENERATION_METRIC_NAMES = ["Answer Relevancy", "Faithfulness", "Correctness", "Completeness"]

print("Metrics defined:")
print(f"  Retrieval: {RETRIEVAL_METRIC_NAMES}")
print(f"  Generation: {GENERATION_METRIC_NAMES}")

Each metric is an LLM-as-a-judge prompt plus parsing logic. DeepEval ships a default prompt for every metric, and what the metric actually measures is a product of that prompt and the model running it.

> 💡 **Tip:** Every DeepEval metric accepts an `evaluation_template` parameter — see the [customize metric prompts documentation](https://deepeval.com/docs/metrics-introduction#customize-metric-prompts). This matters most when your judge model is smaller than DeepEval's defaults assume, when your domain has criteria the defaults weren't designed for, or when you disagree with how DeepEval defines a term like "relevant". Claude Sonnet 4.6 on the default templates is fine for this notebook.

> 💡 **Tip:** The first draft of a G-Eval rarely calibrates well. Run it, read the `reason` field on a few cases, and tighten the steps until the judge's reasoning matches what you'd expect. The Foundational [Quality Metrics module](../../Foundational%20Evaluations/02-quality-metrics/README.md) walks through this calibration loop in detail.

### Checking the harness

Before we run the real evaluation, let's verify the harness works. We call `evaluate()` on two examples with `actual_output` set to the reference answer — every case should pass every metric. If anything fails here, the problem is in the setup, not the models.

This cell is also your first look at `evaluate()`. It takes a list of test cases and a list of metrics and handles concurrency, caching, and result formatting. DeepEval runs metrics asynchronously by default, and we keep that on via `AsyncConfig(run_async=True)` so judge calls across test cases run in parallel.

In [ ]:
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig, DisplayConfig

sanity_cases = [
    LLMTestCase(
        input=r["question"],
        actual_output=r["expected_answer"],  # cheat: use reference as output
        expected_output=r["expected_answer"],
        retrieval_context=[r["context"]],
    )
    for r in rows[:2]
]

ALL_METRICS = RETRIEVAL_METRICS + GENERATION_METRICS

sanity_results = evaluate(
    test_cases=sanity_cases,
    metrics=ALL_METRICS,
    async_config=AsyncConfig(run_async=True),
    display_config=DisplayConfig(print_results=False),
)

passed = sum(1 for tr in sanity_results.test_results if all(m.success for m in tr.metrics_data))
total = len(sanity_cases)
print(f"Harness check: {passed}/{total} cases passed all metrics.")
print("(If this isn't all passing, something is off with the judge or the setup.)")

## 3. Evaluating retrieval quality

Before we generate any model outputs, we evaluate the retrieval context itself. This tells us which examples have strong context (the retriever did its job) and which have weak context (the retriever surfaced something incomplete or off-topic). We need this baseline to interpret generation results later — a model can't produce a good answer from bad context, and we don't want to blame the generator for the retriever's mistakes.

We run the two retrieval metrics on all 20 examples. For this evaluation, we set `actual_output` to the reference answer — `ContextualRelevancyMetric` examines the relationship between `input` and `retrieval_context`; `ContextualRecallMetric` checks whether `retrieval_context` covers the information in `expected_output`.

In [ ]:
# Build test cases for retrieval evaluation.
# actual_output = expected_answer because retrieval metrics focus on
# the relationship between input, retrieval_context, and expected_output.
retrieval_cases = [
    LLMTestCase(
        input=r["question"],
        actual_output=r["expected_answer"],
        expected_output=r["expected_answer"],
        retrieval_context=[r["context"]],
        name=r["id"],
    )
    for r in rows
]

retrieval_results = evaluate(
    test_cases=retrieval_cases,
    metrics=RETRIEVAL_METRICS,
    async_config=AsyncConfig(run_async=True, throttle_value=1),
    display_config=DisplayConfig(print_results=False),
)

# Summarize
for metric_name in RETRIEVAL_METRIC_NAMES:
    passed = sum(
        1 for tr in retrieval_results.test_results
        for md in tr.metrics_data
        if md.name == metric_name and md.success
    )
    print(f"  {metric_name}: {passed}/{len(rows)} passed")

### Interpreting retrieval scores

Low `Contextual Relevancy` means the passage isn't on-topic for the question — the retriever surfaced the wrong chunk entirely. Low `Contextual Recall` means the passage is related but doesn't contain enough information to produce a complete answer.

Let's look at which examples scored lowest on retrieval:

In [ ]:
# Find examples with the weakest retrieval scores.
retrieval_scores = []
for tr in retrieval_results.test_results:
    relevancy = next((m for m in tr.metrics_data if m.name == "Contextual Relevancy"), None)
    recall = next((m for m in tr.metrics_data if m.name == "Contextual Recall"), None)
    retrieval_scores.append({
        "id": tr.name,
        "relevancy": relevancy.score if relevancy else None,
        "recall": recall.score if recall else None,
    })

# Sort by lowest average retrieval score
retrieval_scores.sort(key=lambda x: (x["relevancy"] or 0) + (x["recall"] or 0))

print("Weakest retrieval examples:\n")
for item in retrieval_scores[:5]:
    print(f"  {item['id']:8s}  relevancy={item['relevancy'] or 0:.2f}  recall={item['recall'] or 0:.2f}")

Your results may vary between runs due to LLM-as-a-judge variance, but here's what we saw when we ran this notebook:

- **Contextual Relevancy: 15/20 passed.** The 5 failures were all adjacent-context examples (aws_16–20). For example, aws_16 asks *"What's the maximum size of a single S3 object?"* but the context describes S3's durability, encryption, and versioning — nothing about size limits. The metric correctly scores this 0: the context isn't relevant to the question asked.
- **Contextual Recall: 19/20 passed.** Even the adjacent-context examples scored 1.0 on recall. Why? Our expected answers for those cases say "the context doesn't cover this" — and the metric checks whether the context can support producing that expected answer. Since the expected behavior is to decline, and the context supports declining (by not containing the answer), recall is satisfied. This is a quirk of how the metric interacts with our dataset design, not a bug.

In a production system, consistently low retrieval scores point at upstream problems: the embedding model may not capture domain-specific nuances, the chunk size may be too large or too small, or the document parsing may have lost important structure. DeepEval can't score ingestion directly, but retrieval metrics are the signal that something upstream needs attention.

## 4. Generating candidate outputs

Now that we know which examples have strong context and which have weak or off-topic context, we can generate candidate answers and later interpret generation failures in light of that baseline.

We call each candidate model against every question in the dataset under a single system prompt and collect the responses.

In [ ]:
SYSTEM_PROMPT = (
    "You are a documentation assistant. Answer the user's question using ONLY "
    "the information in the provided context. If the context does not contain "
    "the answer, say so. Respond in 2-3 sentences. Do not add examples or facts "
    "that are not in the context."
)

This prompt is the variable we hold constant across both candidates. It explicitly instructs the model to stay grounded in the provided context and decline when the context doesn't cover the question. Whether models actually follow that instruction — especially on partial-context examples — is exactly what the generation metrics will measure.

In [ ]:
def build_user_prompt(row: dict) -> str:
    return f"Context:\n{row['context']}\n\nQuestion: {row['question']}"


def generate(model_id: str, row: dict) -> str:
    return converse(
        model_id,
        prompt=build_user_prompt(row),
        system=SYSTEM_PROMPT,
        temperature=0,
        max_tokens=400,
    )


# Generate outputs for both candidates.
outputs = {}  # keyed by (model_label, row_id)

for model_label, model_id in [("nova_2_lite", NOVA_2_LITE_ID), ("haiku", HAIKU_ID)]:
    print(f"Generating {model_label} ... ", end="", flush=True)
    for r in rows:
        outputs[(model_label, r["id"])] = generate(model_id, r)
    print("done")

print(f"\nTotal outputs: {len(outputs)} (expected 40)")

Before scoring, let's eyeball a few answers side by side. This helps calibrate expectations — if the outputs already look wrong to you, the metrics should confirm it; if they look fine and the metrics disagree, that's a signal to revisit your metric definitions or thresholds.

In [ ]:
peek_row = rows[1]  # row about AWS Lambda
print(f"Question: {peek_row['question']}\n")
for model_label in ("nova_2_lite", "haiku"):
    text = outputs[(model_label, peek_row["id"])]
    print(f"--- {model_label} ({len(text.split())} words) ---")
    print(text)
    print()

## 5. Evaluating generation quality

With outputs in hand, we score the 40 candidate answers against the four generation metrics. We run one `evaluate()` call per model, keeping the results separate so they're straightforward to compare next.

> 💡 **Note:** If you hit a `ThrottlingException` during this cell, increase the `throttle_value` parameter in `AsyncConfig` (e.g. from 1 to 5). This adds a delay between test cases to stay within your account's token-per-minute quota.

In [ ]:
def build_generation_cases(model_label: str) -> list:
    return [
        LLMTestCase(
            input=r["question"],
            actual_output=outputs[(model_label, r["id"])],
            expected_output=r["expected_answer"],
            retrieval_context=[r["context"]],
            name=f"{model_label}/{r['id']}",
        )
        for r in rows
    ]


generation_results = {}
for model_label in ["nova_2_lite", "haiku"]:
    print(f"Evaluating {model_label} ...")
    cases = build_generation_cases(model_label)
    result = evaluate(
        test_cases=cases,
        metrics=GENERATION_METRICS,
        async_config=AsyncConfig(run_async=True, throttle_value=1),
        display_config=DisplayConfig(print_results=False),
    )
    generation_results[model_label] = result

print("\nDone.")

With both models scored, let's compare their pass rates side by side. The table below shows, for each metric, how many of the 20 examples each model passed at the 0.7 threshold. This is the high-level view — we'll dig into specific failures in the next section.

In [ ]:
# Summarize generation pass rates per model in a comparison table.
print("Generation pass rate per model:\n")
header = f"{'Model':<12}" + "".join(f"{n:<20}" for n in GENERATION_METRIC_NAMES)
print(header)
print("-" * len(header))
for model_label in ["nova_2_lite", "haiku"]:
    cells = []
    for metric_name in GENERATION_METRIC_NAMES:
        passed = total = 0
        for tr in generation_results[model_label].test_results:
            md = next((m for m in tr.metrics_data if m.name.startswith(metric_name)), None)
            if md is None:
                continue
            total += 1
            if md.success:
                passed += 1
        pct = f"{passed}/{total} ({passed/total:.0%})" if total else "n/a"
        cells.append(f"{pct:<20}")
    print(f"{model_label:<12}" + "".join(cells))

### Interpreting generation scores

Your results may vary between runs. Here's what we saw:

| Model | Answer Relevancy | Faithfulness | Correctness | Completeness |
|---|---|---|---|---|
| Nova 2 Lite | 85% | 100% | 80% | 50% |
| Haiku | 80% | 100% | 80% | 65% |

A few observations:

- **Faithfulness is 100% for both models.** The strict prompt ("use ONLY the provided context") keeps both models grounded — they don't invent claims unsupported by the context. The prompt is doing its job.
- **Completeness is the weakest metric (50–65%).** On partial-context examples, both models tend to answer from training knowledge instead of declining as the prompt instructs. For example, aws_09 asks *"How do IAM users differ from IAM roles?"* but the context only describes roles. The expected answer says "the context does not describe IAM users, so a full comparison cannot be made." Both models ignored this and gave a full comparison anyway — scoring 0.10 on completeness. You might expect faithfulness to catch this, but it doesn't: `FaithfulnessMetric` checks whether claims in the output are *supported by* the context, and the model's claims about IAM roles are indeed in the context. It doesn't flag the model for *adding* information the context didn't cover. Completeness catches it instead, because the expected answer's key point ("context doesn't cover IAM users") is missing from the model's output. This is a subtle but important distinction: faithfulness measures grounding of what was said; completeness measures alignment with what *should have been* said.
- **Answer Relevancy and Correctness are strong but not perfect.** A few examples trip up each model, especially on questions where the context is adjacent or partial.

## 6. Diagnosing failures

Scores alone don't tell you what to fix. The key diagnostic skill in RAG evaluation is cross-referencing retrieval and generation results to pinpoint where failures originate — is the problem upstream (bad context) or downstream (bad generation)?

We'll do this in three steps:

1. **Recap both stages side by side** — a quick summary so we don't have to scroll back.
2. **Cross-reference failures** — for each generation failure, was the retrieval context also weak?
3. **Read the judge's reasons** — what specifically went wrong on the worst cases?

In [ ]:
# Recap: retrieval and generation pass rates side by side.
print("=" * 60)
print("RETRIEVAL PASS RATES (from Section 3)")
print("=" * 60)
for metric_name in RETRIEVAL_METRIC_NAMES:
    passed = sum(
        1 for tr in retrieval_results.test_results
        for md in tr.metrics_data
        if md.name == metric_name and md.success
    )
    print(f"  {metric_name:<25} {passed}/{len(rows)} ({passed/len(rows):.0%})")

print(f"\n{'=' * 60}")
print("GENERATION PASS RATES (from Section 5)")
print("=" * 60)
header = f"  {'Model':<12}" + "".join(f"{n:<20}" for n in GENERATION_METRIC_NAMES)
print(header)
print("  " + "-" * (len(header) - 2))
for model_label in ["nova_2_lite", "haiku"]:
    cells = []
    for metric_name in GENERATION_METRIC_NAMES:
        passed = total = 0
        for tr in generation_results[model_label].test_results:
            md = next((m for m in tr.metrics_data if m.name.startswith(metric_name)), None)
            if md is None:
                continue
            total += 1
            if md.success:
                passed += 1
        cells.append(f"{passed}/{total} ({passed/total:.0%})" if total else "n/a")
    print(f"  {model_label:<12}" + "".join(f"{c:<20}" for c in cells))

### Cross-referencing retrieval and generation

The recap above shows *what* is failing. Now we ask *why*: for each generation failure, was the retrieval context also weak?

The logic is simple:
- If retrieval relevancy is low (context is off-topic) AND generation fails → **retrieval problem**. No model can answer well from irrelevant context.
- If retrieval scores are high (context is good) AND generation fails → **generation problem**. The model had what it needed and still got it wrong.
- If retrieval is partial (recall < 1.0) AND generation fails → **mixed cause**. The retriever gave incomplete information, and the model didn't handle the gap gracefully.

The cell below finds all generation failures for both models and shows the retrieval scores for the same examples.

In [ ]:
# Build a lookup of retrieval scores by example ID
retrieval_lookup = {}
for tr in retrieval_results.test_results:
    relevancy = next((m for m in tr.metrics_data if m.name == "Contextual Relevancy"), None)
    recall = next((m for m in tr.metrics_data if m.name == "Contextual Recall"), None)
    retrieval_lookup[tr.name] = {
        "relevancy": relevancy.score if relevancy else None,
        "recall": recall.score if recall else None,
    }

# Find generation failures and cross-reference with retrieval scores.
# Show up to 10 failures per model so both models are always visible.
print("Generation failures cross-referenced with retrieval:\n")
print(f"{'Example':<10} {'Model':<12} {'Failed Metric':<22} {'Score':<8} {'Ret Rel':<10} {'Ret Rec':<10} {'Likely Cause'}")
print("-" * 95)

for model_label in ["nova_2_lite", "haiku"]:
    count = 0
    for tr in generation_results[model_label].test_results:
        for m in tr.metrics_data:
            if not m.success:
                example_id = tr.name.split("/")[-1] if "/" in tr.name else tr.name
                ret = retrieval_lookup.get(example_id, {})
                ret_rel = ret.get('relevancy')
                ret_rec = ret.get('recall')
                # Classify likely cause
                if ret_rel is not None and ret_rel < 0.5:
                    cause = "retrieval"
                elif ret_rel is not None and ret_rel >= 0.7 and (ret_rec is None or ret_rec >= 0.7):
                    cause = "generation"
                else:
                    cause = "mixed"
                print(f"{example_id:<10} {model_label:<12} {m.name:<22} {m.score:<8.2f} "
                      f"{f'{ret_rel:.2f}' if ret_rel is not None else 'n/a':<10} "
                      f"{f'{ret_rec:.2f}' if ret_rec is not None else 'n/a':<10} "
                      f"{cause}")
                count += 1
        if count >= 10:
            break
    if count > 0:
        print()  # blank line between models

if all(
    m.success
    for ml in ["nova_2_lite", "haiku"]
    for tr in generation_results[ml].test_results
    for m in tr.metrics_data
):
    print("No generation failures this run — all cases passed.")

### Reading the judge's reasons

The cross-reference table tells us *where* failures cluster and their likely cause. To understand the specifics, we read the judge's `reason` field on the worst cases — the chain-of-thought explanation the judge model produced when scoring.

> 💡 **Note: why dynamic selection.** LLM-as-a-judge scores vary between runs, so any single example might pass on one run and fail on the next. Picking the weakest cases programmatically each time means the cell stays useful no matter how the scores land.

In [ ]:
# Collect all generation scores across both models
all_gen_scores = []
for model_label in ["nova_2_lite", "haiku"]:
    for tr in generation_results[model_label].test_results:
        for m in tr.metrics_data:
            all_gen_scores.append({
                "name": tr.name,
                "metric": m.name,
                "score": m.score,
                "reason": m.reason,
                "input": tr.input,
                "output": tr.actual_output,
                "expected": tr.expected_output,
            })

# Sort by score ascending, show the three worst
all_gen_scores.sort(key=lambda x: x["score"] if x["score"] is not None else 999)

print("Three lowest-scoring generation cases:\n")
for item in all_gen_scores[:3]:
    print(f"{'=' * 70}")
    print(f"{item['name']} | {item['metric']} | score={item['score']:.2f}")
    print(f"{'=' * 70}")
    print(f"\nQuestion:\n  {item['input']}")
    print(f"\nExpected answer:\n  {item['expected']}")
    print(f"\nModel answer:\n  {item['output']}")
    print(f"\nJudge's reason:\n  {item['reason']}")
    print()

### Making the call — end-to-end diagnosis

From our run, two patterns emerged:

| Example | Retrieval | Generation | Root cause |
|---|---|---|---|
| aws_09 | relevancy=1.0, recall=1.0 | completeness=0.10 | **Generation.** Context was fine, but the model answered from training knowledge instead of declining as instructed. |
| aws_10 | relevancy=0.83, recall=0.67 | completeness=0.10 | **Mixed.** Retrieval gave partial information, and the model didn't handle the gap gracefully. |

Notice that the adjacent-context examples (aws_16–20, where retrieval relevancy=0.0) did *not* appear as generation failures. Both models correctly declined to answer when the context was completely off-topic — the prompt did its job there. The failures cluster on *partial*-context examples, where the context is related enough to tempt the model into answering from training knowledge.

**Diagnostic framework:**

| Retrieval scores | Generation scores | What to fix |
|---|---|---|
| Low | Low | Upstream: chunking strategy, embedding model, or top-K |
| Low | High | Nothing — the model correctly declined when context was irrelevant |
| High | Low | Downstream: prompt wording, few-shot examples, or model choice |
| Partial | Low | Both: improve retrieval coverage AND make the model handle gaps better |
| High | High | Nothing — the system is working as intended |

What "best" means depends on which kind of mistake is worse for your application. A medical assistant that hallucinates is worse than one that declines to answer; a casual FAQ bot that declines too often frustrates users. The metrics give you the signal — the product decision is yours.

## 7. Going further

At this point you've built a complete RAG evaluation pipeline: defined success criteria as metrics, baselined retrieval quality, scored two candidate models on generation, cross-referenced the stages to diagnose root causes, and read the judge's reasoning on failures. This is the workflow you'd repeat whenever you change a model, prompt, retrieval strategy, or dataset.

A few natural next steps from here:

- **Make the model decision.** The scores above give you the signal, but the decision depends on your tolerance for different failure modes. If completeness matters most (users need full answers), the model that scores higher there wins — even if it's slightly worse on another metric. If faithfulness is non-negotiable (e.g. medical or legal), you'd pick the model that never hallucinates, even at the cost of declining more often.
- **Grow the dataset.** 20 examples is a teaching dataset. Real model-selection decisions need a larger dataset ideally drawn from your actual user traffic and expanded whenever you find a new failure mode in production.
- **Add `ContextualPrecisionMetric`** if your pipeline includes a re-ranker and returns multiple passages per query. It evaluates whether relevant passages are ranked above irrelevant ones.
- **Vary the system prompt.** We used one prompt here. Running the same evaluation under different prompting strategies (e.g. a more permissive prompt that allows the model to supplement with general knowledge) reveals how prompt choice affects faithfulness and completeness trade-offs.
- **Lock evaluations into CI.** Once you've picked a model, DeepEval's [pytest integration](https://deepeval.com/docs/evaluation-unit-testing-in-ci-cd) (`deepeval test run`) turns test cases into regression gates that run on every change.

### Resources

- [DeepEval documentation](https://deepeval.com/docs/introduction)
- [DeepEval metrics reference](https://deepeval.com/docs/metrics-introduction)
- [G-Eval](https://deepeval.com/docs/metrics-llm-evals)
- [DeepEval RAG evaluation guide](https://deepeval.com/guides/guides-rag-evaluation)
- [Amazon Bedrock Converse API](https://docs.aws.amazon.com/bedrock/latest/userguide/conversation-inference.html)